# Orbit propagation demo

Расширенная демонстрация моделирования, сравнения с реальными TLE/SGP4 и визуализаций.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Gavr101/space_modeling.git'  # TODO: replace with real repo URL
REPO_DIR = Path('/content/space_modeling') if IN_COLAB else Path.cwd()


def run(cmd):
    print('>>', ' '.join(cmd))
    subprocess.check_call(cmd)

if IN_COLAB:
    if not REPO_DIR.exists():
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
run([sys.executable, '-m', 'pip', 'install', '-e', '.'])
print('Dependencies installed. Working dir:', Path.cwd())

## Что происходит в моделировании

В проекте используются:
- `dynamics.propagator.PropagationConfig` — контейнер параметров запуска интегрирования;
- `dynamics.propagator.propagate_orbit` — численное распространение орбиты;
- `sgp4` — референс/валидационная модель по TLE из CelesTrak;
- `visualization.orbit_3d.build_orbit_figure` — 3D визуализация нескольких траекторий с полупрозрачной Землей;
- `visualization.map_2d.build_groundtrack_figure` — 2D ground-track на карте мира.

Текущий fallback-пропагатор учитывает центральную гравитацию Земли и использует RK4 (вместо неустойчивого Euler). В архитектуре проекта предусмотрено расширение до high-fidelity force models (J2, drag, Sun/Moon, SRP) через TudatPy backend.

### Начальные условия и параметры
Задаются:
- `initial_state = [x, y, z, vx, vy, vz]`;
- `epoch_seconds`, `duration_seconds`, `step_seconds`, `integrator`;
- свойства КА в `spacecraft`: `mass`, `cd`, `cr`, `reference_area`.


In [ ]:
import numpy as np
from datetime import timezone, timedelta

from sgp4.api import Satrec, jday
from sgp4.conveniences import sat_epoch_datetime

from dynamics.propagator import PropagationConfig, propagate_orbit
from visualization.orbit_3d import build_orbit_figure
from visualization.map_2d import build_groundtrack_figure

In [ ]:
TLES = {
    'ISS (ZARYA)': (
        '1 25544U 98067A   26128.52749306  .00016178  00000+0  29081-3 0  9998',
        '2 25544  51.6389 262.3902 0003522 120.6870 311.5993 15.50359005504712',
    ),
    'STARLINK-1008': (
        '1 44714U 19074B   26128.32137731  .00003179  00000+0  23789-3 0  9991',
        '2 44714  53.0540 170.1880 0001221  99.9037 260.2108 15.06387939356311',
    ),
}

In [ ]:
def sgp4_state_samples(name, tle1, tle2, duration_hours=8, step_seconds=60):
    sat = Satrec.twoline2rv(tle1, tle2)
    epoch = sat_epoch_datetime(sat).replace(tzinfo=timezone.utc)

    samples = []
    times = []
    for dt_s in range(0, int(duration_hours * 3600) + 1, step_seconds):
        t = epoch + timedelta(seconds=dt_s)
        jd, fr = jday(t.year, t.month, t.day, t.hour, t.minute, t.second + t.microsecond / 1e6)
        e, r_km, v_km_s = sat.sgp4(jd, fr)
        if e != 0:
            raise RuntimeError(f'SGP4 error for {name}: {e}')
        samples.append(np.array([*r_km, *v_km_s], dtype=float))
        times.append(t)

    states = np.array(samples)
    states[:, :3] *= 1000.0
    states[:, 3:] *= 1000.0
    return np.array(times), states


def run_model_from_initial_sgp4_state(initial_state, duration_hours=8, step_seconds=60):
    cfg = PropagationConfig(
        initial_state=initial_state,
        epoch_seconds=0.0,
        duration_seconds=duration_hours * 3600,
        step_seconds=step_seconds,
        integrator='DOP853',
    )
    _, model_states = propagate_orbit(cfg)
    return model_states

In [ ]:
tracks_eci = []
names_3d = []

for sat_name, (tle1, tle2) in TLES.items():
    _, real_states = sgp4_state_samples(sat_name, tle1, tle2, duration_hours=6, step_seconds=60)
    model_states = run_model_from_initial_sgp4_state(real_states[0], duration_hours=6, step_seconds=60)

    tracks_eci.append(real_states[:, :3])
    names_3d.append(f'{sat_name} | SGP4')
    tracks_eci.append(model_states[:, :3])
    names_3d.append(f'{sat_name} | Model')

    pos_err = np.linalg.norm(model_states[:, :3] - real_states[:, :3], axis=1)
    vel_err = np.linalg.norm(model_states[:, 3:] - real_states[:, 3:], axis=1)
    print(f'{sat_name}: max |Δr| = {pos_err.max()/1000:.1f} km, max |Δv| = {vel_err.max():.2f} m/s')

In [ ]:
fig3d = build_orbit_figure(states=tracks_eci, names=names_3d, show_earth=True)
fig3d.show()

## 2D карта траекторий

Для вставки в презентации удобнее ground-track. Ниже — приближённое отображение на карте (используется преобразование ECEF→lat/lon; для строгого ground-track нужен поворот ECI→ECEF по времени).

In [ ]:
fig2d = build_groundtrack_figure(tracks_eci, names=names_3d)
fig2d.show()